In [1]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

import torch
# import transformers
from datetime import datetime
from pathlib import Path

from transformers import AutoModelForCausalLM, AutoTokenizer

from music21 import stream, note
import re

from collections import Counter


In [ ]:
last_modified = datetime.fromtimestamp(
    Path("examuse_hw.ipynb").stat().st_mtime
)

In [5]:
last_modified

datetime.datetime(2026, 1, 28, 15, 25, 41, 174258)

*Last updated:* {{ last_modified.strftime("%Y-%m-%d") }}

*Created:* {{< meta date >}}  
*Last updated:* {{ last_modified.strftime("%Y-%m-%d") }}


# Stage 1. Paper Summary

## Title, authors

- **Title**: Practical and Reproducible Symbolic Music Generation by Large Language Models with Structural Embeddings
- **Authors**: Seungyeon Rhyu, Kichang Yang, Sungjun Cho, Jaehyeon Kim, Kyogu Lee, Moontae Lee
- [ArXiv preprint](https://arxiv.org/abs/2407.19900)
- Submitted on 29 Jul 2024

## Description

Large language models should be naturally extended to generate symbolic music. However, some music-specific aspects turn out to be challenging for those models. In particular, MIDI data often lack annotations of bars and beats, a fact that impedes effective tokenization. Heuristic algorithms trying to generate such annotations cannot be applied reliably because in most of the MIDI data generated from real music audio, performers start or end notes at points deviating from the scripted tempo. 

Music-generation frameworks like MuseNet [@huangMusicTransformer2018] introduced additional *structural embeddings* to extract the structural context of music from MIDI files. However, according to Rhyu et al. [-@rhyuPracticalReproducibleSymbolic2024], their choices were not well-argumented and the experiments they made were not well-documented. These limitations hinder further testing, and potential application to data that lacks domain annotations. 

To address these limitations, Rhyu et al.:

- train a vanilla GPT-2 model
- implement the structural embeddings proposed by MuseNet
- perform ablation studies for different initialization methods

The overarching aim is two fold:

1. Gain insights into how to effectively encode structural information without manual annotations
2. Provide an accessible pipeline to train models on large music datasets. 

## Summary of the main ideas / achievements / challenges / future work 

**Big research question** 
- How to effectively toekenize MIDI data that often lack annotations of bars and beats

**Smaller research question**, directly addressed by Rhyu et al. 
- Implement reproducible structural embeddings.

**Achievements**
- Reproducible structural tokenization
- Online demonstration of the generated music

**Notable Results**
- Structural embeddings are beneficial for symbolic music generation

**Challenges and future work**
- Better objective metrics are needed to capture the structureness of data beyond simple repetition. 


## Key Terms 

- **Symbollic Music Generation** - training DL models to generate music based on discrete tokens (MIDI, MusicXML) as opposed to raw waveforms.  

- **Large Language Models** - fundamental DL models based on the transformer architecture trained on massive, unstructured datasets to understand and generate human-like text

- **Structural Embeddings** - vector representations that explicitly capture the structural relationships or hierarchical organization of data points. Rhyu et al. use 4 types of structural embeddings:
    - Part (out of 128 subdivisions of each song)
    - Type of token
    - Time
    - Pitch Class

- **Initialization Methods** - methods to initialize structura embeddings 
1. truncated normal initialization
2. sinusoidal initialization, applied only to the time-related embeddings 

- **Evaluation Metrics** of musical quality
	- objective  
	- subjective 


# Stage 2: Architecture and key results (usually figure / table) 

## Architecture

- Backbone architecture: GPT-2

- No sparse attention and mixup 

- Concatenate structural embeddings with input token embeddings

- Positional encoding to the resulting embeddings 

### Training

- uses next-token prediction task. 

- At each step, predict $x_i$ from $x_{<i}$ by optimizing <paste loss fn> 

- Training uses next-token prediction task. 
	- At each step, predict $x_i$ from $x_{<i}$ by optimizing 
    $J = -\sum{\log p(x_i | x_{<i})} $
    This loss fucntion is cross-entropy. 


The architecture & training procedure are illustrated in Figure 3, left panel: 


![Figure 3, left](Fig3_left.png)

### Inference 

- the model autoregressively generates the sequence given a prompt of MIDI tokens
- after each token is generated, its four structural labels are extracted using rule-based modules

Illustrated in Figure 3, right panel: 

![Figure 3, right](Fig3_right.png)

### Initialization Methods

Structural embeddings are initialized in one of two ways: 

- Truncated normal initalizatin (also called random initialization in the text)

- Sinusiodal Initialization, following Vaswani et al. [-@vaswaniAttentionAllYou2017] and Guo et al. [-@guoDomainknowledgeinspiredMusicEmbedding2023]
    - $SE_{(k, 2i)} = \sin (k / (10000 / w)^{2i/d}) $
    - $SE_{(k, 2i+1)} = \cos (k / (10000 / w)^{2i/d}) $


where: 
- $k$ is a class index for part or time, 
- $i$ is a feature index, 
- $d$ is the hidden size, and 
- $w$ is a scaling factor. 

Using two different values for $w$ ensures that the corresponding embeddings are orthogonal, allowing the part and time attributes to be represented without interference.

## Evaluation Metrics

### Objective evaluation 

- _Structureness indicator_ (SI): measures the largest degree of repeatedness among various intervals in generated music; a fitness score based on a similarity matrix

- _Chord Progression Validation Rationality_ (CPVR): conditional probability of the existence of each unique chord n-gram given its previous n-gram within the generated music.

- _Chord Progression Irregularity_ (CPI): measures the ratio of unique chord n-grams from generated music, compensating CPVR that yields high scores with frequently occurring chords

### Subjective Evaluation 

- A-vs-B human-rating procedure. 

- Four human raters

- Rating scales:

    - Naturalness
    
    - Prompt maintenance

## Key Results

- Adding structural embeddings to GPT2 enhances the ability of the model to capture structural aspects of music
- Random initialization genarates music that sounds better to the human ear than sinusoidal initialization
- Sinusoidal initialization of temporal embeddings produces more repeated patterns and common chords 



- **Fitness Scape Plots**
	- Context: how to find the most representative segment of a musical piece? 
	- [Müller et al.](https://www.audiolabs-erlangen.de/resources/MIR/FMP/C4/C4S3_AudioThumbnailing.html) propose a fitness measure assigning a fitness value of each musical segment. The fitness measure is the harmonic mean of two aspects: 
		1. how well a given segment explains other related segments, and 
		2. how much of the overall music recording is covered by all these related segments. 
	
		According to this metric, the segment of maximal fitness is the most representative part of a song. 
	- Fitness Scape Plots [@mullerScapePlotRepresentation2012], (see also [this online notebook](https://www.audiolabs-erlangen.de/resources/MIR/FMP/C4/C4S3_ScapePlot.html)) are triangular representations of musical segments, with center of segment on the $X$ axis and duration on the $Y$ axis. 
	- With the chosen colormap (`cmap = hot_r`), darker colors represent higher fitness values. 
    - Fitness Scape Plots in the paper (Figure 6) reveal a difference between ground truth and GPT-generated melodies: ground truth may
feature musical patterns in longer durations relative to the GPT-generated music. 

![Figure 6: Fitness Scape Plots](Fig6_fitness_scape_plots.png)

# Stage 3: Reproduction and ideas

## Reproduction

### Setup

The [huggingface page of EAM](https://huggingface.co/acl-submission-anonym) containts two models: realtive and spectral. These are also the names that are referred to on the [GitHub repo](https://github.com/anonymous-submission-351532/examuse-transformers). The names used in the paper, GPT2-RE & GPT-SE, are lacking. I deduce that the spectral model loosly corresponds to GPT2-SE, and the relative model to GPT2-RE, and compare them, along with a baseline GPT2 model. 

My aim is to reproduce a part of Table 1 from Rhyu et al, showing the resulst of the objective evaluation scores:

![Table 1: Objective Evaluation Score](Table_1.png)


I first run a minimal test that the models load and run, then proceed to reproducing the model comparison. 

#### Minmal Tests that models load and run

- Load

In [ ]:
model_name = "acl-submission-anonym/EAM-spectral"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

model.eval()

- Generate music

In [ ]:
prompt = "<bos> <v64_C4> <time_2>"

inputs = tokenizer(prompt, return_tensors="pt")

with torch.no_grad():
    output = model.generate(
        **inputs,
        max_time_length=[50],
        max_length=100,
        do_sample=False,
        top_k=50,
        temperature=1.0
    )

generated = tokenizer.decode(output[0])
print(generated)

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


<pad> <v64_C4> <time_2> <v0_C4> <time_0> <v0_C3> <time_0> <v0_C1> <v0_C2> <time_0> <v0_C1> <time_99> <time_99> </s> <s> <time_99> </s> <s> <time_99> </s> <s> <time_99> </s> <s> <time_99> </s> <s> <time_99> </s> <s> <time_99> </s> <s> <time_99> </s> <s> <time_99> </s> <s> <time_99> </s> <s> <time_99> </s> <s> <time_99> </s> <s> <time_99> </s> <s> <time_99> </s> <s> <time_99> </s> <s> <time_99> </s> <s> <time_99> </s> <s> <time_99> </s> <s> <time_99> </s> <s> <time_99> </s> <s> <time_99> </s> <s> <time_99> </s> <s> <time_99> </s> <s> <time_99> </s> <s> <time_99> </s> <s> <time_99> </s> <s> <time_99> </s> <s> <time_99> </s> <s> <time_99> </s> <s> <time_99>


The minimal test ran succesfully, so I proceed to the reproduction. 



#### Compare the three models



- MIDI Converter

Since the models output tokens, I convert the outputs to MIDI files that are saved in the working
directory and can be viewed or heard using external software (e.g., MuseScore).

In [13]:
def eam_tokens_to_midi(token_string, output_file="output.mid", time_step=0.25, max_time_val=4):
    """
    Converts EAM-style tokens to a MIDI file.
    Clips long time steps to reduce rests.
    """
    s = stream.Stream()
    current_offset = 0.0
    tokens = token_string.split()

    for tok in tokens:
        # Match note tokens like <v64_C4>
        match = re.match(r"<v(\d+)_(\w+\d)>", tok)
        if match:
            velocity = int(match.group(1))
            pitch = match.group(2)
            n = note.Note(pitch)
            n.volume.velocity = velocity
            n.quarterLength = time_step
            n.offset = current_offset
            s.insert(current_offset, n)
        # Match time tokens like <time_2>
        elif tok.startswith("<time_"):
            time_val = int(tok.replace("<time_", "").replace(">", ""))
            time_val = min(time_val, max_time_val)  # clip excessive rests
            current_offset += time_val * time_step
        # Ignore other tokens like <bos>

    s.write("midi", fp=output_file)
    # s.show("midi")
    print(f"Saved MIDI to {output_file}")

* Structureness Indicator (SI)

The largest degree of repeatedness among various intervals in generated music

In [23]:
def structureness_indicator(tokens, min_interval=3, max_interval=15):
    pitches = []
    for t in tokens:
        if t.startswith("<v"):
            m = re.match(r"<v\d+_(\w[#b]?\d)>", t)  # allow sharps/flats
            if m:
                pitches.append(m.group(1))
            # else: skip token
    N = len(pitches)
    si_scores = []
    for start in range(N):
        for interval in range(min_interval, max_interval+1):
            if start + 2*interval > N:
                continue
            seq1 = pitches[start:start+interval]
            seq2 = pitches[start+interval:start+2*interval]
            if seq1 == seq2:
                si_scores.append(interval)
    return max(si_scores)/N if si_scores else 0.0

- Chord Extraction 

This is needed for the metrics related to chords -- Chord Progression Validation Rationality (CPVR) and Chord Progression Irregularity (CPI):

In [15]:
def extract_chords(tokens, time_step=0.25, window=1.0):
    chords = []
    current_chord = []
    offset = 0.0
    for tok in tokens:
        if tok.startswith("<v"):
            m = re.match(r"<v\d+_(\w[#b]?\d)>", tok)
            if m:
                pitch = m.group(1)
                current_chord.append(pitch)
        elif tok.startswith("<time_"):
            time_val = int(tok.replace("<time_","").replace(">",""))
            offset += time_val * time_step
            if offset >= window:
                if current_chord:
                    chords.append(tuple(sorted(current_chord)))
                    current_chord = []
                offset = 0.0
    if current_chord:
        chords.append(tuple(sorted(current_chord)))
    return chords

- Chord metrics (CPI & CPVR)

In [16]:
def chord_metrics(chords, n=2):
    N = len(chords)
    ngrams = [tuple(chords[i:i+n]) for i in range(N-n)]
    unique_ngrams = set(ngrams)
    cpi = len(unique_ngrams) / max(1, len(ngrams))
    prev_to_next = Counter()
    prev_count = Counter()
    for i in range(len(ngrams)-1):
        prev = ngrams[i][:-1]
        next_chord = ngrams[i][-1]
        prev_to_next[(prev, next_chord)] += 1
        prev_count[prev] += 1
    cpvr = sum(v/prev_count[k[0]] for k,v in prev_to_next.items()) / len(prev_count) if prev_count else 0.0
    return cpvr, cpi

- Generation + Evaluation with multiple prompt lengths

To reproduce Rhyu et al., I need a function capable of handling multiple prompt lengths: 

> we take the beginning of a song as a prompt and examine the prompt token lengths of 2l with l ∈ {4, 6, 8} (The lengths in time are about 2, 5, 15 seconds, respectively). We use top-k sampling with k = 32 until the sample reaches 1 minute maximum.



In [17]:
def generate_music_from_prompt(model_name, full_prompt_tokens, max_tokens=300, top_k=32, temperature=1.0, midi_prefix="sample"):
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForCausalLM.from_pretrained(model_name)
    model.eval()

    prompt_str = "<bos> " + " ".join(full_prompt_tokens)
    inputs = tokenizer(prompt_str, return_tensors="pt")

    with torch.no_grad():
        if "EAM" in model_name:
            output = model.generate(
                **inputs,
                max_length=max_tokens,
                do_sample=True,
                top_k=top_k,
                temperature=temperature,
                max_time_length=[max_tokens]
            )
        else:
            output = model.generate(
                **inputs,
                max_length=max_tokens,
                do_sample=True,
                top_k=top_k,
                temperature=temperature
            )

    gen_tokens = tokenizer.decode(output[0]).split()
    midi_file = f"{midi_prefix}.mid"
    eam_tokens_to_midi(" ".join(gen_tokens), midi_file)

    # Compute metrics
    si = structureness_indicator(gen_tokens)
    chords = extract_chords(gen_tokens)
    cpvr_cpi = {n: chord_metrics(chords, n=n) for n in [2,3,4]}

    return gen_tokens, si, cpvr_cpi

I inspect models with prompts of $2^6 = 64$ tokens. 

-> Prepare $2^6 = 64$ token prompt

In [18]:
long_prompt_tokens = []
pitches = ["C4","D4","E4","F4","G4","A4","B4","C5"]
velocities = [60,62,64,65,67,69,70,72]
time_steps = [1,2,1,2]

for i in range(32):  # 32*2 = 64 tokens
    pitch = pitches[i % len(pitches)]
    vel = velocities[i % len(velocities)]
    time = time_steps[i % len(time_steps)]
    long_prompt_tokens.append(f"<v{vel}_{pitch}>")
    long_prompt_tokens.append(f"<time_{time}>")

print(f"Prompt length: {len(long_prompt_tokens)} tokens")


Prompt length: 64 tokens


Compare three models

In [19]:
models = {
    "GPT-2": "gpt2",
    "EAM-relative": "acl-submission-anonym/EAM-relative",
    "EAM-spectral": "acl-submission-anonym/EAM-spectral"
}

final_results = {}

for name, model_name in models.items():
    print(f"\n=== Generating for {name} ===")
    gen_tokens, si, cpvr_cpi = generate_music_from_prompt(
        model_name,
        long_prompt_tokens,
        max_tokens=300,
        top_k=32,
        temperature=1.0,
        midi_prefix=name
    )
    final_results[name] = {"Tokens": gen_tokens, "SI": si, "ChordMetrics": cpvr_cpi}


=== Generating for GPT-2 ===


Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Input length of input_ids is 387, but `max_length` is set to 300. This can lead to unexpected behavior. You should consider increasing `max_new_tokens`.


Saved MIDI to GPT-2.mid

=== Generating for EAM-relative ===


Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Saved MIDI to EAM-relative.mid

=== Generating for EAM-spectral ===


Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Saved MIDI to EAM-spectral.mid


### Results

In [21]:
# Print aligned summary table
header = f"{'Model':<12} {'PromptTokens':<13} {'SI':<6} {'CPVR2':<7} {'CPI2':<7} {'CPVR3':<7} {'CPI3':<7} {'CPVR4':<7} {'CPI4':<7}"
print("\n=== Summary Table ===")
print(header)
print("-"*len(header))

for model, metrics in final_results.items():
    cpvr_cpi = metrics["ChordMetrics"]
    print(f"{model:<12} {len(long_prompt_tokens):<13} "
          f"{metrics['SI']:<6.3f} "
          f"{cpvr_cpi[2][0]:<7.3f} {cpvr_cpi[2][1]:<7.3f} "
          f"{cpvr_cpi[3][0]:<7.3f} {cpvr_cpi[3][1]:<7.3f} "
          f"{cpvr_cpi[4][0]:<7.3f} {cpvr_cpi[4][1]:<7.3f}")



=== Summary Table ===
Model        PromptTokens  SI     CPVR2   CPI2    CPVR3   CPI3    CPVR4   CPI4   
---------------------------------------------------------------------------------
GPT-2        64            0.250  1.000   0.889   1.000   1.000   1.000   1.000  
EAM-relative 64            0.040  1.000   0.960   1.000   1.000   1.000   1.000  
EAM-spectral 64            0.121  1.000   0.776   1.000   0.938   1.000   1.000  


### Reproduction: Discussion

The present results are very different from those on Rhyu et al.'s Table 1. Most likely, this is due to combination of reasons such as:

- Their results are produced on many prompts
- Their prompts were segments from real musical pieces
- The indices might not be calculated properly in this project
- Other hyperparameters might not be well aligned
- Information not reported by the authors. 


## Ideas for extensions

- The work is limited to solo piano. Adding more instruments might be a promising new direction
- As the authors themselves note, a good musical piece contains repetitions of special patterns (not just a given prompt)
A future challenge would be to generate music that contains such musical motives
- Relatedly, the model can be expanded to jazz/blues improvisation. In combination with motive creation, this could be a 
good avenue for future AI generation of musical improvisation. 

# References
